### Fine-tune on Colab

In [ ]:
# @title Run parameters
import os

# Browser or VS Code Colab: uncomment these lines and set the values
# Colab CLI: keep them commented and pass the values with `colab exec --env KEY=VALUE`
# %env MODEL=Qwen/Qwen2.5-Coder-1.5B-Instruct
# %env CONFIG=configs/qlora.yaml

names = ("MODEL", "CONFIG")
missing = [name for name in names if name not in os.environ]
if missing:
    raise RuntimeError(
        f"Set {', '.join(missing)}: colab exec --env KEY=VALUE, or %env KEY=VALUE in a cell above"
    )

MODEL, CONFIG = os.environ["MODEL"], os.environ["CONFIG"]

REPO = "unbiasederudite/qwen2.5-coder-text2sql-qlora"  # GitHub repository
REF = "main"  # branch or tag
REPO_DIR = REPO.split("/")[-1]
os.environ.update(REPO=REPO, REF=REF, REPO_DIR=REPO_DIR)

In [ ]:
# @title Install the project and download Spider
import codecs
import subprocess

script = """
set -eu
curl -fsSL https://raw.githubusercontent.com/$REPO/$REF/scripts/colab_setup.sh -o /tmp/colab_setup.sh
bash /tmp/colab_setup.sh
bash "/content/$REPO_DIR/scripts/download_data.sh" spider
"""
decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
with subprocess.Popen(
    ["bash", "-c", script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT
) as process:
    while chunk := process.stdout.read1(1024):
        print(decoder.decode(chunk), end="", flush=True)
if process.returncode:
    raise RuntimeError(f"setup failed with exit code {process.returncode}")

In [ ]:
# @title Imports
%cd -q /content/{REPO_DIR}

import tqdm.auto as tqdm_auto
from tqdm import tqdm

tqdm_auto.tqdm = tqdm

from datetime import UTC, datetime
from pathlib import Path

import transformers.trainer as hf_trainer

from text2sql.data import build_samples
from text2sql.meta import metadata_path, write_training_metadata
from text2sql.model import MODELS, load_model, model_slug
from text2sql.spider import SPIDER_DIR
from text2sql.train import (
    MetricsProgressCallback,
    has_checkpoint,
    is_finished,
    load_config,
    train,
)

hf_trainer.DEFAULT_PROGRESS_CALLBACK = MetricsProgressCallback

The training config, the samples of its Spider training splits, and the model with its tokenizer loaded in 4-bit.

In [ ]:
# @title Load the samples and the model
config = load_config(Path(CONFIG))
if MODEL not in MODELS:
    raise ValueError(f"MODEL must be one of {MODELS}, not {MODEL!r}")

samples = [
    sample for split in config["splits"] for sample in build_samples(SPIDER_DIR, split, clean=True)
]
model, tokenizer = load_model(MODEL)

LoRA fine-tuning on the SQL of each sample. The adapter is saved to the directory below. A rerun resumes from the last epoch, and does nothing if the adapter is finished.

In [ ]:
# @title Fine-tune the model
adapter_dir = Path("adapters_colab") / model_slug(MODEL)
needs_training = not is_finished(adapter_dir)
resumed = has_checkpoint(adapter_dir)
started_at = datetime.now(UTC)
if needs_training:
    counts = train(model, tokenizer, samples, adapter_dir, config)
    print(f"trained adapter on {counts['samples']} samples in {adapter_dir}")
else:
    print(f"nothing was trained, the adapter is already in {adapter_dir}")
finished_at = datetime.now(UTC)

Metadata of the run, saved with the adapter.

In [ ]:
# @title Write the run metadata
if needs_training:
    written = write_training_metadata(
        adapter_dir,
        model,
        config=config,
        counts=counts,
        resumed=resumed,
        started_at=started_at,
        finished_at=finished_at,
    )
    print(f"metadata in {written}")
else:
    print(f"metadata kept in {metadata_path(adapter_dir)}")